In [5]:
 # ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from imblearn.over_sampling import SMOTE

# ================================
# 2. LOAD DATA
# ================================
df = pd.read_csv("online_retail.csv", encoding='ISO-8859-1')

# ================================
# 3. DATA CLEANING
# ================================
df = df.dropna()

df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df = df[df['UnitPrice'] > 0]
df = df.drop_duplicates()

# ================================
# 4. FEATURE ENGINEERING
# ================================
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
df['is_return'] = df['Quantity'] < 0

customer_df = df.groupby('CustomerID').agg({
    'InvoiceNo': 'count',
    'TotalAmount': 'sum',
    'is_return': 'sum',
    'InvoiceDate': ['min', 'max']
})

customer_df.columns = [
    'total_orders',
    'total_spent',
    'total_returns',
    'first_order',
    'last_order'
]

customer_df = customer_df.reset_index()

# behavioral features
customer_df['return_ratio'] = customer_df['total_returns'] / customer_df['total_orders']

customer_df['customer_lifetime_days'] = (
    customer_df['last_order'] - customer_df['first_order']
).dt.days

customer_df['customer_lifetime_days'] = customer_df['customer_lifetime_days'].replace(0, 1)

customer_df['avg_order_value'] = customer_df['total_spent'] / customer_df['total_orders']

customer_df['purchase_frequency'] = customer_df['total_orders'] / customer_df['customer_lifetime_days']

customer_df = customer_df.fillna(0)

# ================================
# 5. CREATE FRAUD LABEL
# ================================
customer_df['fraud'] = (
    (customer_df['return_ratio'] > 0.5) |
    (customer_df['total_returns'] > 10)
).astype(int)

print("Fraud Distribution:\n", customer_df['fraud'].value_counts())

# ================================
# 6. MODEL TRAINING
# ================================
X = customer_df[['total_orders', 'total_spent', 'total_returns',
                 'return_ratio', 'customer_lifetime_days',
                 'avg_order_value', 'purchase_frequency']]

y = customer_df['fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ================================
# 7. HANDLE CLASS IMBALANCE (SMOTE)
# ================================
smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)

# ================================
# 8. TRAIN MODEL
# ================================
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ================================
# 9. EVALUATION
# ================================
y_pred = model.predict(X_test)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ================================
# 10. FEATURE IMPORTANCE
# ================================
importance = model.feature_importances_
features = X.columns

feat_imp = pd.DataFrame({
    'Feature': features,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importance:\n")
print(feat_imp)

# ================================
# 11. RISK SCORE (0–100)
# ================================
prob = model.predict_proba(X_test)[:,1]
risk_score = prob * 100

print("\nSample Risk Scores:\n", risk_score[:10])

# ================================
# 12. SAVE MODEL
# ================================
pickle.dump(model, open('model.pkl', 'wb'))

print("\nModel saved as model.pkl ✅")

Fraud Distribution:
 fraud
0    4137
1     234
Name: count, dtype: int64

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       831
           1       0.98      0.98      0.98        44

    accuracy                           1.00       875
   macro avg       0.99      0.99      0.99       875
weighted avg       1.00      1.00      1.00       875


Feature Importance:

                  Feature  Importance
2           total_returns    0.350585
3            return_ratio    0.317857
1             total_spent    0.113885
0            total_orders    0.088982
5         avg_order_value    0.086819
4  customer_lifetime_days    0.022293
6      purchase_frequency    0.019579

Sample Risk Scores:
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Model saved as model.pkl ✅
